In [ ]:
import pandas as pd
import os
from google.colab import drive

# 1. 掛載 Google Drive
drive.mount('/content/drive')

# 2. 設定檔案路徑 (請確保資料夾名稱與檔名正確)
# 根據你的描述，路徑設為「金融資料探勘」資料夾
folder_path = '/content/drive/MyDrive/金融資料探勘/'
file_name = '期中報告資料來源.csv'  # 如果雲端檔案是「期中報告資料來源.csv」，請自行更修
full_path = os.path.join(folder_path, file_name)

try:
    # 3. 讀取 CSV
    # 注意：收盤價可能含有逗號 (如 "18,270.51")，需處理
    df = pd.read_csv(full_path)

    # 4. 重新命名欄位
    df = df.rename(columns={
        '年月日': 'Date',
        '收盤價(元)': 'S0'
    })

    # 5. 資料清洗與轉型
    # 將 S0 中的逗號移除並轉為浮點數
    if df['S0'].dtype == 'object':
        df['S0'] = df['S0'].str.replace(',', '').astype(float)

    # 將 Date 轉為日期格式，確保能正確提取年、月、日
    df['Date_dt'] = pd.to_datetime(df['Date'])

    # 6. 生成 File 欄位
    # 格式要求：OptionsDaily_YYYY_MM_DD.csv (月份與日期需補零)
    df['File'] = df['Date_dt'].dt.strftime('OptionsDaily_%Y_%m_%d.csv')

    # 7. 整理最後的 DataFrame (只保留需要的欄位並調整順序)
    # 假設你想要跟 Path_教學_0305 類似的結構
    result_df = df[['Date', 'File', 'S0']]

    # 顯示前五筆結果
    print("處理完成！前五筆資料如下：")
    print(result_df.head())

    # 8. (選填) 將結果存成新的 CSV 檔案回雲端
    # result_df.to_csv(os.path.join(folder_path, 'Processed_Path_Data.csv'), index=False)

except FileNotFoundError:
    print(f"錯誤：在 {folder_path} 找不到檔案 {file_name}，請檢查雲端硬碟路徑。")
except Exception as e:
    print(f"發生錯誤：{e}")

In [ ]:
import pandas as pd
import datetime
import calendar
import os
from google.colab import drive, files

# 1. 掛載 Google Drive
drive.mount('/content/drive')

# 2. 設定路徑與檔名
folder_path = '/content/drive/MyDrive/金融資料探勘/'
input_file = '期中報告資料來源.csv'  # 請確保檔案存在於該路徑
full_path = os.path.join(folder_path, input_file)

def get_third_wednesday(year, month):
    """計算指定年月的第一個星期三"""
    c = calendar.monthcalendar(year, month)
    first_week = c[0]
    second_week = c[1]
    third_week = c[2]
    fourth_week = c[3]

    # 判斷第三個星期三在哪一週
    if first_week[calendar.WEDNESDAY]:
        return datetime.date(year, month, third_week[calendar.WEDNESDAY])
    else:
        return datetime.date(year, month, fourth_week[calendar.WEDNESDAY])

def get_rf_rate(date):
    """
    對照 2022 年台灣銀行一年期定期儲蓄存款(一般、機動利率)
    0.84% (至 3/20), 1.09% (3/21-6/19), 1.215% (6/20-9/25), 1.34% (9/26-12/18), 1.465% (12/19起)
    """
    d = date.date() if isinstance(date, datetime.datetime) else date
    if d < datetime.date(2022, 3, 21):
        return 0.84
    elif d < datetime.date(2022, 6, 20):
        return 1.09
    elif d < datetime.date(2022, 9, 26):
        return 1.215
    elif d < datetime.date(2022, 12, 19):
        return 1.34
    else:
        return 1.465

try:
    # 3. 讀取並清洗資料
    df = pd.read_csv(full_path)
    df = df.rename(columns={'年月日': 'Date', '收盤價(元)': 'S0'})

    # 處理 S0 逗號並轉為浮點數
    if df['S0'].dtype == 'object':
        df['S0'] = df['S0'].str.replace(',', '').astype(float)

    # 轉換 Date 為 datetime 格式
    df['Date_dt'] = pd.to_datetime(df['Date'])

    # 4. 生成 File 欄位
    df['File'] = df['Date_dt'].dt.strftime('OptionsDaily_%Y_%m_%d.csv')

    # 5. 計算結算日、到期天數與合約月份
    expiry_dates = []
    maturities = []
    contracts = []

    for current_date in df['Date_dt']:
        curr_d = current_date.date()

        # 尋找目前的結算日 (本月第三個星期三)
        expiry = get_third_wednesday(curr_d.year, curr_d.month)

        # 若交易日 >= 結算日 (要求 Maturity >= 1)，則切換到下個月
        if curr_d >= expiry:
            next_month = curr_d.month + 1
            year = curr_d.year
            if next_month > 12:
                next_month = 1
                year += 1
            expiry = get_third_wednesday(year, next_month)

        diff = (expiry - curr_d).days
        expiry_dates.append(expiry)
        maturities.append(diff)
        contracts.append(expiry.strftime('%Y%m'))

    df['ContractExpiryDate'] = expiry_dates
    df['Maturity'] = maturities
    df['Contract'] = contracts

    # 6. 加入 Rf (利率)
    df['Rf'] = df['Date_dt'].apply(get_rf_rate)

    # 7. 整理最後欄位
    final_df = df[['Date', 'File', 'S0', 'Maturity', 'Contract', 'ContractExpiryDate', 'Rf']]

    # 8. 儲存並下載
    output_filename = '修正後_期中報告資料.csv'
    final_df.to_csv(output_filename, index=False, encoding='utf-8-sig')
    print("處理完成！檔案已生成。")
    files.download(output_filename)

except FileNotFoundError:
    print(f"找不到檔案：{full_path}，請檢查雲端硬碟路徑。")
except Exception as e:
    print(f"錯誤：{e}")